In [1]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response

In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

"With 200,000 inhabitants in 1328, Paris, then already the capital of France, was the most populous city of Europe. By comparison, London in 1300 had 80,000 inhabitants.[29] Around 1340 the city had between 250,000-300,000 inhabitants.[30] By the early fourteenth century... Capital of France. Paris is the capital of France, and therefore is the seat of France's national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. Paris, the cosmopolitan capital of France, has the reputation of being the most beautiful and romantic of all cities, brimming with historic associations and remaining vastly influential in the realms of culture, art, fashion, food and design. The Panthéon honors some of France’s greatest figures, while the Cluny baths and the ancient arenas of Lutetia reveal traces of the Roman past. Walking along Rue Mouffetard, the medieval character of the capital becomes visible in its charming façades a

In [5]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response

In [7]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American businessman and entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence research organization OpenAI since 2019.\nAltman attended Stanford University for two years before he dropped out and co-founded Loopt, a smartphone geosocial networking service, which raised more than $30 million in venture capital before being acquired by Green Dot Corporation for $43.4 million in cash. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. \nAfter co-founding OpenAI in 2015, Altman later became the organization\'s CEO in 2019. In 2023, he was ousted by the organization\'s board of directors, who cited a lack of "confidence in his ability to continue leading OpenAI" in an official post. The move was met with significant backlash from employees and investors, resulting in Altman\'s reins

In [8]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)

In [11]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [12]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

In [13]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content

In [14]:
tool_rag.invoke("When was NovaSphere Organization founded?")

/var/folders/cr/34qx8njd48g07ps00m4fshq40000gn/T/ipykernel_11570/1495361009.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

In [15]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag
]

In [16]:
llm_bind = llm.bind_tools(toolkit)

In [17]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 315, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzOZeQClRGpQeqkEq2K9ql46GjB1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0a3-2b33-7c40-8ec2-4a7af694f6da-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_0UjNPcX3EQjxiyGXoCnEXEUA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 26, 'total_tokens': 341, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## ReAct Agent

In [18]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [19]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='8868fae9-fc2f-4f31-8a5e-c68acf674fdf'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 315, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzPNyPqQ5FofI5JEZQjhbPwyfhcw', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0a3-f1ed-7dd3-b616-5e28c459a454-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_5Dh3YlDVebnRCn9uRi9BdoU3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [20]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

{'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='5e1c684a-d181-4fc6-98cc-8be149854862'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 309, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DVzPhXrB0PUxRHkYpa5zOsTqHhxVG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da0a4-402d-7453-9aa2-366c463b8d60-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_fNXadkJeUzx8fuWdVmlETEtB', 'type': 'tool_call'}], invalid_tool_calls=[], usage_